In [ ]:
# ==============================================================================
# R|S ATLAS INGESTION: STEP 1
# 
# ARCHITECTURE NOTE: The R|S Atlas ontology is provided as a static HTML table 
# embedded in a journal article. This script uses Strategy A (Local Bulk 
# Parsing) via BeautifulSoup on a local HTML file.
#
# SSSOM ALIGNMENT STRATEGY: 
# 1. Root Nodes: Bolded elements (<strong>) serve as skos:Collection root nodes.
# 2. Child Nodes: Indented elements serve as skos:Concept child nodes.
# 3. CURIEs: Synthesized sequentially (e.g., RSATLAS:1, RSATLAS:1A).
# 4. Data Loss: Statistical counts of items/constructs from the native table 
#    are intentionally dropped to maintain semantic structural purity.
# ==============================================================================
# INSTRUCTIONS FOR REPRODUCIBILITY
# 1. Navigate to the published article: https://doi.org/10.1136/bmjopen-2020-043830#T3 
# 2. Inspect the DOM and copy the raw HTML for Table 3.
# 3. Save the snippet as `rs_atlas_table3.html` in your local `data/external/` directory.
# ==============================================================================

import os
import sys
import re
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

# --- 1. Environment & Central Schema Setup ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "raw"))
external_data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "external"))
os.makedirs(raw_data_dir, exist_ok=True)

load_dotenv(os.path.join(config_dir, ".env")) 
sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    raise ImportError("CRITICAL: Could not find ingest_schema_manager.py. Check PYTHONPATH.")

# --- 2. Registry Lookup & Target Setup ---
SOURCE_PREFIX = "RSATLAS"
registry_data = get_registry_info(
    prefix=SOURCE_PREFIX, 
    config_dir=config_dir, 
    fallback_name="R|S Atlas",
    fallback_uri="" # Synthetic CURIEs, no base URI
)
SOURCE_NAME = registry_data['Source_Name']
raw_ingest_file = os.path.join(raw_data_dir, f"raw_{SOURCE_PREFIX.lower()}.csv")
html_path = os.path.join(external_data_dir, "rs_atlas_table3.html")

# --- 3. Persistent Tracking Check ---
if os.path.exists(raw_ingest_file):
    print(f"[*] Overwriting previous {os.path.basename(raw_ingest_file)} due to local bulk extraction.")
    os.remove(raw_ingest_file)

# --- 4. Helper Functions ---
def clean_label(text):
    """Strips whitespace, unicode indents (\u2003), and newlines."""
    if not text: return ""
    text = text.replace('\u2003', '').replace('\n', ' ')
    return re.sub(r'\s+', ' ', text).strip()

def write_row(data_dict):
    clean_row = finalize_row(data_dict)
    pd.DataFrame([clean_row])[COLUMN_ORDER].to_csv(
        raw_ingest_file, mode='a', index=False, 
        header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
    )

# --- 5. Main Extraction Loop ---
try:
    if not os.path.exists(html_path):
        print(f"[!] Critical Error: Could not find R|S Atlas HTML at {html_path}")
        sys.exit(1)

    print(f"[*] Loading Document: {os.path.basename(html_path)}...")
    with open(html_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')
    
    tbody = soup.find('tbody')
    if not tbody:
        print("[!] Critical Error: No <tbody> found in the HTML table.")
        sys.exit(1)
        
    rows = tbody.find_all('tr')
    print(f"[*] Found primary taxonomy table with {len(rows)} rows.")
    
    current_cat_name = ""
    current_cat_id = 0
    spec_idx = 0
    total_processed = 0
    
    for row in rows:
        tds = row.find_all('td')
        if not tds: continue
        
        label_td = tds[0]
        raw_text = label_td.get_text()
        clean_text = clean_label(raw_text)
        
        if not clean_text: continue
        
        # 5a. Detect Parent Node (via <strong> tag)
        if label_td.find('strong'):
            current_cat_id += 1
            current_cat_name = clean_text
            spec_idx = 0
            
            write_row({
                'Source_System': SOURCE_NAME,
                'Primary_Label': current_cat_name,
                'CURIE': f"{SOURCE_PREFIX}:{current_cat_id}",
                'Formal_Label': "",
                'Concept_Type': 'skos:Collection',
                'Hierarchy_Path': current_cat_name,
                'Description': "",
                'Parent_IDs': "",
                'Concept_ID': str(current_cat_id),
                'Status': "active",
            })
            total_processed += 1
            print(f"\r[Ingesting] Root: {current_cat_name[:40]:<40} (ID: {current_cat_id})", end="", flush=True)
            
        # 5b. Detect Child Node (lacks <strong> tag)
        else:
            spec_char = chr(65 + spec_idx) # Converts 0 -> A, 1 -> B, etc.
            spec_id_str = f"{current_cat_id}{spec_char}"
            
            write_row({
                'Source_System': SOURCE_NAME,
                'Primary_Label': clean_text,
                'CURIE': f"{SOURCE_PREFIX}:{spec_id_str}",
                'Formal_Label': "",
                'Concept_Type': 'skos:Concept',
                'Hierarchy_Path': f"{current_cat_name} > {clean_text}",
                'Description': "",
                'Parent_IDs': str(current_cat_id),
                'Concept_ID': spec_id_str,
                'Status': "active",
            })
            total_processed += 1
            spec_idx += 1
            
except Exception as e:
    print(f"\n[!] Error during parsing: {e}")

print(f"\n\n[COMPLETE] Extracted {total_processed} R|S Atlas concepts to Bronze Layer.")